## GET DATASETS AND PREPROCESS

In [ ]:
from jwst_acquire import search, inspect_products, search_by_program
from jwst_acquire import download_chips, download_by_index, load_bands, TARGETS

obs = search(TARGETS["carina"])
i2d = inspect_products(obs)
i2d2 = search_by_program("5408")  
paths = download_chips(i2d, preferred_detector="nrca1")
bands = load_bands(paths)


────────────────────────────────────────────────────────────
  Searching: Cosmic Cliffs — Carina Nebula
  Coord    : 161.23750°  -59.86667°
  Radius   : 0.05°
────────────────────────────────────────────────────────────
  Found 2 observations

                obs_id                target_name ... t_obs_release 
------------------------------------- ----------- ... --------------
jw05408-o024_t026_nircam_f150w2-f162m      HH-900 ... 61244.62758098
jw05408-o024_t026_nircam_f150w2-f164n      HH-900 ... 61244.62758098

  #    filename                                                          type
  ──── ───────────────────────────────────────────────────────────────── ──────────────
  [0 ] jw05408024001_02101_00001_nrca3_i2d.fits                          chip (detector)  119 MB
  [1 ] jw05408024001_02101_00001_nrca2_i2d.fits                          chip (detector)  118 MB
  [2 ] jw05408024001_02101_00001_nrcb2_i2d.fits                          chip (detector)  119 MB
  [3 ] jw05408024001_

TimeoutError: Timeout limit of 600 exceeded.

In [ ]:
from astroquery.mast import Observations
from pathlib import Path
from astropy.io import fits
from jwst_acquire import search, inspect_products, search_by_program
from jwst_acquire import download_chips, download_by_index, load_bands, TARGETS
obs = search(TARGETS["carina"])
i2d = inspect_products(obs)
manifest = Observations.download_products(i2d[[4]], download_dir="./jwst_data")

# Robust path find — searches the whole folder
fname   = Path(manifest["Local Path"][0]).name
matches = list(Path("./jwst_data").rglob(fname))
p       = matches[0]
print(f"Found at: {p}")

# Check what filter it is
with fits.open(p) as hdul:
    phdr = hdul[0].header
    print("FILTER  :", phdr.get("FILTER"))
    print("PUPIL   :", phdr.get("PUPIL"))
    print("TARGNAME:", phdr.get("TARGNAME"))
    print("PROGRAM :", phdr.get("PROGRAM"))


────────────────────────────────────────────────────────────
  Searching: Cosmic Cliffs — Carina Nebula
  Coord    : 161.23750°  -59.86667°
  Radius   : 0.05°
────────────────────────────────────────────────────────────
  Found 2 observations

                obs_id                target_name ... t_obs_release 
------------------------------------- ----------- ... --------------
jw05408-o024_t026_nircam_f150w2-f162m      HH-900 ... 61244.62758098
jw05408-o024_t026_nircam_f150w2-f164n      HH-900 ... 61244.62758098

  #    filename                                                          type
  ──── ───────────────────────────────────────────────────────────────── ──────────────
  [0 ] jw05408024001_02101_00001_nrca3_i2d.fits                          chip (detector)  119 MB
  [1 ] jw05408024001_02101_00001_nrca2_i2d.fits                          chip (detector)  118 MB
  [2 ] jw05408024001_02101_00001_nrcb2_i2d.fits                          chip (detector)  119 MB
  [3 ] jw05408024001_

IndexError: list index out of range

In [ ]:
from astropy.coordinates import SkyCoord
from astroquery.mast import Observations
import astropy.units as u

# Try a wider radius and no program filter
coord = SkyCoord("10h45m53s -59d52m20s", frame="icrs")

obs = Observations.query_criteria(
    coordinates    = coord,
    radius         = 0.2 * u.deg,        # wider net
    obs_collection = "JWST",
    instrument_name= "NIRCAM/IMAGE",
    calib_level    = 3,
)

print(f"Found {len(obs)} observations\n")
print(obs["obs_id", "target_name", "proposal_id", "filters", "t_obs_release"])

Found 28 observations

                obs_id                    target_name     ... t_obs_release 
------------------------------------- ------------------- ... --------------
 jw05408-o004_t005_nircam_f444w-f470n              HH-903 ... 61203.07721061
jw05408-o022_t024_nircam_f150w2-f162m              HH1006 ... 61202.80412024
 jw05408-o005_t006_nircam_clear-f444w NAME-TREASURE-CHEST ... 61166.52886571
jw05408-o024_t026_nircam_f150w2-f164n              HH-900 ... 61244.62758098
jw05408-o024_t026_nircam_f150w2-f162m              HH-900 ... 61244.62758098
 jw05408-o024_t026_nircam_clear-f444w              HH-900 ... 61244.62758098
jw05408-o004_t005_nircam_f150w2-f162m              HH-903 ... 61203.07721061
jw05408-o004_t005_nircam_f150w2-f164n              HH-903 ... 61203.07721061
 jw05408-o020_t020_nircam_f444w-f470n             HH-1008 ... 61207.11695605
jw05408-o020_t020_nircam_f150w2-f162m             HH-1008 ... 61207.11695605
                                  ...                

In [ ]:
from astroquery.mast import Observations
for name in ["CARINA*", "COSMIC*CLIFFS*", "NGC*3324*", "CAR-I*"]:
    obs = Observations.query_criteria(
        target_name    = name,
        obs_collection = "JWST",
        instrument_name= "NIRCAM/IMAGE",
        calib_level    = 3,
    )
    if len(obs) > 0:
        from astropy.table import unique
        s = unique(obs["proposal_id","target_name","filters"], keys="proposal_id")
        print(f"\n'{name}' → {len(obs)} obs")
        print(s)
    else:
        print(f"'{name}' → nothing")

'CARINA*' → nothing
'COSMIC*CLIFFS*' → nothing

'NGC*3324*' → 6 obs
proposal_id target_name filters
----------- ----------- -------
       2731    NGC-3324   F187N
'CAR-I*' → nothing


In [ ]:
targets = {
    "Pillars": SkyCoord("18h18m48s -13d49m00s", frame="icrs"),
    "SN1987A": SkyCoord("05h35m28s -69d16m11s", frame="icrs"),
    "Tarantula": SkyCoord("05h38m38s -69d05m42s", frame="icrs"),
}

for label, coord in targets.items():
    obs = Observations.query_criteria(
        coordinates    = coord,
        radius         = 0.1 * u.deg,
        obs_collection = "JWST",
        instrument_name= "NIRCAM/IMAGE",
        calib_level    = 3,
    )
    if len(obs) > 0:
        from astropy.table import unique
        s = unique(obs["proposal_id","target_name","filters"], keys="proposal_id")
        print(f"\n{label} → {len(obs)} obs")
        print(s)
    else:
        print(f"{label} → nothing found")


Pillars → 6 obs
proposal_id target_name filters
----------- ----------- -------
       2739        M-16   F200W

SN1987A → 37 obs
proposal_id target_name   filters   
----------- ----------- ------------
       1232    SN-1987A        F335M
       1726    SN-1987A F322W2;F323N
       7575    SN-1987A        F210M

Tarantula → 16 obs
proposal_id      target_name      filters
----------- --------------------- -------
       1226 47-eMPTselected+allTA   F187N
       2729              NGC-2070   F187N


 Getting oberservation for 2731 (Carina Nebula)

In [ ]:
obs_2731 = Observations.query_criteria(
    target_name    = "NGC*3324*",
    obs_collection = "JWST",
    instrument_name= "NIRCAM/IMAGE",
    calib_level    = 3,
    proposal_id    = "2731",
)

print(f"Total observations: {len(obs_2731)}")
print(obs_2731["obs_id", "target_name", "filters", "t_exptime"])

Total observations: 6
               obs_id                target_name   filters       t_exptime     
------------------------------------ ----------- ----------- ------------------
jw02731-o001_t017_nircam_clear-f187n    NGC-3324       F187N            5797.86
jw02731-o001_t017_nircam_clear-f335m    NGC-3324       F335M 3221.0400000000004
jw02731-o001_t017_nircam_clear-f200w    NGC-3324       F200W 3221.0400000000004
jw02731-o001_t017_nircam_clear-f090w    NGC-3324       F090W 3221.0400000000004
jw02731-o001_t017_nircam_clear-f444w    NGC-3324       F444W 3221.0400000000004
jw02731-o001_t017_nircam_f444w-f470n    NGC-3324 F444W;F470N            5797.86


In [ ]:
from astroquery.mast import Observations

products_2731 = Observations.get_product_list(obs_2731)
i2d_2731 = Observations.filter_products(
    products_2731,
    productSubGroupDescription="I2D",
    extension="fits",
)

# Print with sizes
print(f"\n{'#':<4} {'filename':<65} {'MB':>8}")
print(f"{'─'*4} {'─'*65} {'─'*8}")
for i, row in enumerate(i2d_2731):
    name     = row["productFilename"]
    size_mb  = (row["size"] or 0) / 1e6
    is_chip  = any(d in name for d in 
                   ["nrca1","nrca2","nrca3","nrca4","nrcb1","nrcb2","nrcb3","nrcb4"])
    kind     = "chip" if is_chip else "MOSAIC"
    print(f"[{i:<2}] {name:<65} {size_mb:>8.0f}   {kind}")


#    filename                                                                MB
──── ───────────────────────────────────────────────────────────────── ────────
[0 ] jw02731001001_02105_00005_nrcb1_i2d.fits                               118   chip
[1 ] jw02731001001_02105_00004_nrca3_i2d.fits                               119   chip
[2 ] jw02731001001_02101_00002_nrca3_i2d.fits                               119   chip
[3 ] jw02731001001_02101_00005_nrcalong_i2d.fits                            120   MOSAIC
[4 ] jw02731001001_02101_00002_nrca2_i2d.fits                               118   chip
[5 ] jw02731001001_02101_00002_nrcb4_i2d.fits                               119   chip
[6 ] jw02731001001_02101_00004_nrcblong_i2d.fits                            120   MOSAIC
[7 ] jw02731001002_02101_00003_nrcblong_i2d.fits                            120   MOSAIC
[8 ] jw02731001001_02103_00003_nrcb3_i2d.fits                               119   chip
[9 ] jw02731001001_02103_00001_nrcb3_i2d.fits     

Getting Observation for SN1987A (Pillars of Creation)

In [ ]:
from astroquery.mast import Observations
from astropy.coordinates import SkyCoord
import astropy.units as u

obs_sn = Observations.query_criteria(
    coordinates    = SkyCoord("05h35m28s -69d16m11s", frame="icrs"),
    radius         = 0.05 * u.deg,
    obs_collection = "JWST",
    instrument_name= "NIRCAM/IMAGE",
    calib_level    = 3,
)

print(f"Total: {len(obs_sn)} observations\n")
print(obs_sn["obs_id", "proposal_id", "filters", "t_exptime", "t_obs_release"])

Total: 20 observations

                   obs_id                    proposal_id ... t_obs_release 
-------------------------------------------- ----------- ... --------------
        jw07575-o001_t001_nircam_clear-f210m        7575 ... 61315.66283564
 jw01726-o001_t001_nircam_clear-f150w-sub320        1726 ... 60191.04019672
jw01726-o001_t001_nircam_f322w2-f323n-sub320        1726 ... 60191.04019672
 jw01726-o001_t001_nircam_clear-f200w-sub320        1726 ... 60191.04019672
 jw01726-o001_t001_nircam_f405n-f444w-sub320        1726 ... 60191.04019672
        jw07575-o003_t001_nircam_clear-f444w        7575 ... 61322.13839115
 jw01726-o001_t001_nircam_clear-f212n-sub320        1726 ... 60191.04019672
        jw07575-o003_t001_nircam_clear-f200w        7575 ... 61322.13839115
        jw07575-o001_t001_nircam_clear-f200w        7575 ... 61315.66283564
        jw07575-o003_t001_nircam_clear-f210m        7575 ... 61322.13839115
        jw07575-o003_t001_nircam_clear-f356w        7575 ... 613

In [12]:
products_sn = Observations.get_product_list(obs_sn)
i2d_sn = Observations.filter_products(
    products_sn,
    productSubGroupDescription="I2D",
    extension="fits",
)

print(f"\n{'#':<4} {'filename':<65} {'MB':>8}  {'type'}")
print(f"{'─'*4} {'─'*65} {'─'*8}  {'─'*8}")
for i, row in enumerate(i2d_sn):
    name    = row["productFilename"]
    size_mb = (row["size"] or 0) / 1e6
    is_chip = any(d in name for d in
                  ["nrca1","nrca2","nrca3","nrca4","nrcb1","nrcb2","nrcb3","nrcb4"])
    kind    = "chip" if is_chip else "MOSAIC"
    print(f"[{i:<2}] {name:<65} {size_mb:>8.0f}   {kind}")


#    filename                                                                MB  type
──── ───────────────────────────────────────────────────────────────── ────────  ────────
[0 ] jw01726001001_03101_00005_nrcb1_i2d.fits                                 3   chip
[1 ] jw01726001001_03101_00003_nrcb1_i2d.fits                                 3   chip
[2 ] jw01726001001_03101_00002_nrcb1_i2d.fits                                 3   chip
[3 ] jw01726001001_03101_00008_nrcb1_i2d.fits                                 3   chip
[4 ] jw01726001001_03101_00008_nrcb3_i2d.fits                                 3   chip
[5 ] jw01726001001_03101_00002_nrcb2_i2d.fits                                 3   chip
[6 ] jw01726001001_03101_00004_nrcb3_i2d.fits                                 3   chip
[7 ] jw01726001001_03101_00005_nrcb2_i2d.fits                                 3   chip
[8 ] jw01726001001_03101_00004_nrcb2_i2d.fits                                 3   chip
[9 ] jw01726001001_03101_00001_nrcb1_i2d

In [ ]:
# PLOT BANDS
from pathlib import Path
from jwst_acquire import load_bands, plot_bands
import numpy as np

# Point directly at downloaded files — no astroquery needed
fits_files = list(Path("./jwst_data/sn1987a").glob("*i2d.fits"))
print("Files found:", [f.name for f in fits_files])

bands = load_bands(fits_files, rgb_filters=["f150w", "f200w", "f444w"])
plot_bands(bands)

Files found: ['jw01232-o001_t001_nircam_clear-f115w_i2d.fits', 'jw01232-o001_t001_nircam_clear-f277w_i2d.fits', 'jw01232-o001_t001_nircam_clear-f444w_i2d.fits']

Loading F444W...


KeyError: "Extension 'DQ' not found."